# SOXX 对 QQQ 下跌日 beta 缺口：来源与训练去重预检

固定前60个完整交易日，信号当日收盘不参与；只查来源覆盖与原39输入相关，不读取未来标签。


In [1]:
import os,sys,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings
card=root/'research/stage1_soxx_qqq_downside_beta_source_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_model_fits','new_native_accounts','future_outcome_reads'))
qqq=root/'data/snapshots/stage1_soxx_qqq_downside_beta_source_20261023/QQQ_chart.json';soxx=root/src['SOXX_Yahoo_raw'];parent=root/src['original39_card']
assert sha(soxx)==src['SOXX_Yahoo_raw_sha256'] and sha(root/src['SOXX_Yahoo_card']/'summary.json')==src['SOXX_Yahoo_summary_sha256']
assert sha(parent/'summary.json')==src['original39_summary_sha256'] and sha(parent/'qlib_recorder'/src['original39_experiment_id']/src['original39_recorder_id']/'artifacts/inputs/features')==src['original39_features_artifact_sha256']
assert sha(root/src['Qlib_calendar_provider']/'qlib.yaml')==src['Qlib_config_sha256']
t=json.loads((card/'transport_result.json').read_text());a=t['attempt'];assert t['plan_sha256']==sha(card/'plan.json') and t['GETs_consumed']==1 and t['retries']==0 and a['http_status']==200 and a['body_sha256']==sha(qqq) and a['body_bytes']==qqq.stat().st_size and a['path']==plan['exact_GET']['path'] and a['query']==plan['exact_GET']['query']
source_receipt=json.loads((card/'nas_source_archive.json').read_text());assert source_receipt['verified'] and source_receipt['members'][str(qqq.relative_to(root))]==sha(qqq)
old_receipt=json.loads((root/src['SOXX_Yahoo_card']/'nas_source_archive.json').read_text());assert old_receipt['verified'] and old_receipt['sha256']==src['SOXX_Yahoo_NAS_sha256']
def parse(path,symbol):
 item=json.loads(path.read_text())['chart']['result'][0];m=item['meta'];assert m['symbol']==symbol and m['currency']=='USD' and m['exchangeTimezoneName']=='America/New_York'
 q=item['indicators']['quote'][0];dates=pd.to_datetime(item['timestamp'],unit='s',utc=True).tz_convert('America/New_York').date
 x=pd.DataFrame({'date':pd.to_datetime(dates),'open':q['open'],'high':q['high'],'low':q['low'],'close':q['close'],'volume':q['volume']}).set_index('date')
 assert x.index.is_unique and x.index.is_monotonic_increasing and np.isfinite(x[['open','high','low','close']]).all().all() and (x[['open','high','low','close']]>0).all().all()
 assert (x.low<=x[['open','close','high']].min(axis=1)).all() and (x.high>=x[['open','close','low']].max(axis=1)).all()
 return x,{k:len(v) for k,v in item.get('events',{}).items()}
xs,es=parse(soxx,'SOXX');xq,eq=parse(qqq,'QQQ')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_calendar_provider']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'));assert len(cal)==3966
missing_soxx=cal.difference(xs.index);missing_qqq=cal.difference(xq.index)
assert len(missing_soxx)==0 and len(missing_qqq)==0
prices=pd.DataFrame({'SOXX_close':xs.loc[cal,'close'],'QQQ_close':xq.loc[cal,'close']});rets=prices.pct_change(fill_method=None)
N=len(cal);gap=np.full(N,np.nan);beta_all=np.full(N,np.nan);beta_down=np.full(N,np.nan);n_down=np.zeros(N,dtype=int)
for i in range(60,N):
 a=rets.iloc[i-60:i].to_numpy(dtype=float)
 if not np.isfinite(a).all():continue
 rs,rq=a[:,0],a[:,1];mask=rq<0;n_down[i]=int(mask.sum())
 if n_down[i]<15 or np.var(rq,ddof=1)<=0 or np.var(rq[mask],ddof=1)<=0:continue
 beta_all[i]=np.cov(rs,rq,ddof=1)[0,1]/np.var(rq,ddof=1)
 beta_down[i]=np.cov(rs[mask],rq[mask],ddof=1)[0,1]/np.var(rq[mask],ddof=1)
 gap[i]=beta_down[i]-beta_all[i]
frame=pd.DataFrame({'signal_date':cal,'downside_beta_gap_60':gap,'beta_all_60':beta_all,'beta_down_60':beta_down,'negative_QQQ_days_60':n_down}).set_index('signal_date')
train=frame.loc['2011-01-03':'2022-12-30'];held=frame.loc['2023-01-03':'2026-10-09']
assert len(train)==3020 and len(held)==946 and frame.loc['2018-01-02':'2022-12-30','downside_beta_gap_60'].notna().all() and held.downside_beta_gap_60.notna().all()
ps=json.loads((parent/'summary.json').read_text());assert ps['recorder_id']==src['original39_recorder_id']
with R.uri_context(ps['recorder_uri']):features=R.get_recorder(recorder_id=src['original39_recorder_id'],experiment_id=src['original39_experiment_id']).load_object('inputs/features')
original=ps['input_audit']['original_features'];assert len(original)==39 and all(col in features.columns for col in original)
paired=features.loc['2011-01-03':'2022-12-30',original].join(frame[['downside_beta_gap_60']],how='inner')
nov={}
for col in original:
 x=paired[[col,'downside_beta_gap_60']].replace([np.inf,-np.inf],np.nan).dropna()
 nov[col]={'rows':int(len(x)),'Pearson':float(x[col].corr(x.downside_beta_gap_60)) if len(x)>10 and x[col].nunique()>1 else None}
valid={k:abs(v['Pearson']) for k,v in nov.items() if v['Pearson'] is not None};max_name=max(valid,key=valid.get)
report={'decision':'conditional_cross_market_downside_beta_source_map_complete_novelty_descriptive_no_outcomes','plan_sha256':sha(card/'plan.json'),'QQQ_raw_rows':len(xq),'SOXX_raw_rows':len(xs),'QQQ_events_in_response':eq,'SOXX_events_in_response':es,'Qlib_calendar_rows':len(cal),'missing_QQQ_dates':len(missing_qqq),'missing_SOXX_dates':len(missing_soxx),'train_rows':len(train),'train_known_coordinate_rows':int(train.downside_beta_gap_60.notna().sum()),'held_rows':len(held),'held_known_coordinate_rows':int(held.downside_beta_gap_60.notna().sum()),'2018_2022_all_known':True,'coordinate_all_known_summary':{'median':float(frame.downside_beta_gap_60.median()),'p05':float(frame.downside_beta_gap_60.quantile(.05)),'p95':float(frame.downside_beta_gap_60.quantile(.95))},'min_negative_QQQ_days_on_known':int(frame.loc[frame.downside_beta_gap_60.notna(),'negative_QQQ_days_60'].min()),'max_abs_original39_training_Pearson':{'feature':max_name,'absolute_value':valid[max_name],'signed_value':nov[max_name]['Pearson'],'paired_rows':nov[max_name]['rows']},'historical_first_receipt_qualified':False,'new_model_fits':0,'new_native_accounts':0,'future_outcome_reads':0}
static=pd.concat({'feature':frame[['downside_beta_gap_60','beta_all_60','beta_down_60']],'meta':frame[['negative_QQQ_days_60']]},axis=1);static.index=pd.MultiIndex.from_arrays([cal,['SOXX_QQQ']*N],names=['datetime','instrument']);ds=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':('2011-01-03','2022-12-30'),'held':('2023-01-03','2026-10-09')});assert len(ds.prepare('train',col_set=['feature','meta']))==3020 and len(ds.prepare('held',col_set=['feature','meta']))==946
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);ds.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds,source_hashes={'QQQ':sha(qqq),'SOXX':sha(soxx),'original39':src['original39_features_artifact_sha256']});rec.save_objects(artifact_path='analysis',source_frame=frame,training_novelty=nov,report=report);rec.log_metrics(train_known=report['train_known_coordinate_rows'],held_known=report['held_known_coordinate_rows'],max_abs_training_Pearson=valid[max_name],model_fits=0,native_accounts=0)
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':report['plan_sha256'],'source_archive':'nas_source_archive.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'result':'result.json','new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Yahoo current revised close histories are not certified historical first receipts or price vintages.','A distinct formula and training correlation do not imply incremental future tail-risk information.','No tail label, fit, account or SOXX economic gate was read.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='soxx-qqq-downbeta-') as tmp:
 archive=Path(tmp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/report') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'train_known':report['train_known_coordinate_rows'],'held_known':report['held_known_coordinate_rows'],'max_corr':report['max_abs_original39_training_Pearson'],'recorder_id':rid},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[25342:MainThread](2026-10-23 07:59:12,041) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[25342:MainThread](2026-10-23 07:59:12,043) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[25342:MainThread](2026-10-23 07:59:12,043) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


...

/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[25342:MainThread](2026-10-23 07:59:12,313) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name stage1_soxx_qqq_downside_beta_source_20261023.


[25342:MainThread](2026-10-23 07:59:12,315) INFO - qlib.workflow - [exp.py:258] - Experiment 107662550415421774 starts running ...


[25342:MainThread](2026-10-23 07:59:12,371) INFO - qlib.workflow - [recorder.py:345] - Recorder e357303114444af79ae34c41f4b58e41 starts running under Experiment 107662550415421774 ...


[25342:MainThread](2026-10-23 07:59:13,522) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{
  "train_known": 2959,
  "held_known": 946,
  "max_corr": {
    "feature": "range_pos_252",
    "absolute_value": 0.323765933362613,
    "signed_value": 0.323765933362613,
    "paired_rows": 2959
  },
  "recorder_id": "e357303114444af79ae34c41f4b58e41"
}


# SOXX 对 QQQ 下跌日 beta 缺口：来源与去重门

固定使用信号日之前 60 个完整 Qlib 交易日的 SOXX、QQQ 原始收盘简单收益。`下跌 beta 缺口 = QQQ 负收益日上的 SOXX/QQQ 回归斜率 − 全部 60 日斜率`，至少要求 15 个 QQQ 下跌日；没有读取信号日收盘或未来标签。它检验“半导体对广义科技跌势的额外敏感度”，与此前 SOXL 对 SOXX 下跌 beta、原 39 项广度/离散度坐标不同，但目前只是来源假说。

一次冻结 Yahoo QQQ 请求与已封存 SOXX 原始收盘来源各有 3,974 个日线行；2011-01-03 至 2026-10-09 的 3,966 个 Qlib 交易日两源均齐全。最初 61 个训练日因 60 日预热未知，其余 2011–2022 训练日 **2,959/3,020** 已知；2018–2022 全部已知；2023–2026 复用段 **946/946** 已知。已知窗口最少仍有 17 个 QQQ 负收益日。与原 Stage1 39 项在训练配对行上的最大绝对 Pearson 为 **0.324**（`range_pos_252`），没有明显单列复制。

原始 QQQ 与 SOXX 响应、冻结计划、Qlib DatasetH/Recorder 已分别按 SHA-256 归档 Yanxuan NAS 并冷读。Yahoo 当前修订历史及最初接收时点未认证；去重相关性低也不证明未来风险预测有效。0 未来 SOXX 标签读取、0 模型拟合、0 账户。

**决定：来源覆盖与机械去重通过，可另行冻结条件性尾部风险信息检验；不形成交易特征，不推广。** 通过信息门后仍需认证历史时点、完整 Stage1 0bp Tiered 自有账户和合同 v2 的剩余门。
